In [ ]:
pip install youtube-transcript-api

In [ ]:
pip install -U langchain-text-splitters

In [15]:
pip install -qU langchain-huggingface


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install langchain-community

In [ ]:
pip install sentence-transformers

In [ ]:
pip install langchain langchain-huggingface huggingface_hub

## Docloader

In [1]:
from youtube_transcript_api import YouTubeTranscriptApi

In [46]:
video_id = "dDkynerzV-Q"

transcripter = YouTubeTranscriptApi()
transcript = transcripter.fetch(video_id)

In [47]:
data = " ".join([line.text for line in transcript])
# print(data)

## Text splitter

In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.create_documents([data])

## Embedding and vector store

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [53]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(texts, embeddings)

In [54]:
vector_store.index_to_docstore_id

{0: '4c9bc9bd-a69a-4889-bf29-412b41eda439',
 1: 'f9c230ac-fd0d-4839-b890-ec92489222c8',
 2: 'f6a9426c-580e-4a63-ba24-567aa83eea73',
 3: 'c209197b-53b1-4546-8ffe-0084368b0989',
 4: 'd5d6ca03-39f7-45b1-92b2-f27d9723f1ab',
 5: '74621eaf-e5d5-4e9f-85fb-259b5da8509c',
 6: 'c1fff6fa-e01e-4d48-8e78-a350114ebbe3',
 7: 'a221f913-f60d-4d7f-915e-164008128011',
 8: '966e0fbd-9983-4df0-9d18-27b49a501004',
 9: '57c1d197-9fea-4de2-8f78-46012adf2287',
 10: '53155b50-3d25-4ce8-9595-de146af20918',
 11: '5db3b0b5-af53-4df9-bef6-61e29ce8499b',
 12: '3d112d71-8255-4259-86c5-fed9e252a071',
 13: '6673727a-269d-479b-9bfd-d5e36df397f1',
 14: 'f8a5abc0-35d6-4179-b450-ea641a2ab38b',
 15: '6eaf908c-88ed-4c28-9353-3c348b1bd306',
 16: '8f692297-75cf-43f9-9b0a-1e92a7527f5a'}

## Retriever

In [57]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5})

In [ ]:
retriever.invoke("what is rag")

## Augmentation

In [97]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-8B",
    max_new_tokens=512,
)

llm = ChatHuggingFace(llm=llm)

In [42]:
from langchain_core.prompts import PromptTemplate

In [63]:
promptTemplate = PromptTemplate(template = """
    You are a helpful ai assistant, answer the question only using the context provided below:
    {context}
    
    if not possible say don't know

    question: {question}
    """, input_variables=['context','question'])

In [64]:
question = "what is rag"
context = "\n".join([doc.page_content for doc in retriever.invoke(question)])

In [66]:
prompt = promptTemplate.invoke({"context":context, "question":question})

In [67]:
llm.invoke(prompt)

' answer: rag is a simple way to provide a general answer to a question. RAG is a technique where you can provide a few key words in your question and ask the model to provide a general answer. It is based on the idea that the model will extract relevant information from the text and provide a general answer. There are many different approaches to implementing rag. Some use the TF-IDF technique to calculate the relevance of the keywords in the question. Others use the BM25 algorithm to calculate the relevance of the keywords in the question. Still others use the cosine similarity to calculate the relevance of the keywords in the question. The key advantage of rag is that it provides a general answer to a question. It does not require the model to understand the details of the question and can provide a good answer. The key disadvantage of rag is that it can be difficult to optimize for it can be difficult to get good results for rag because it is a simple technique.\n'

In [69]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

In [73]:
def formatDocs(docs):
    return "\n".join([doc.page_content for doc in docs])

In [74]:
parellel = RunnableParallel({
    'context': retriever | RunnableLambda(formatDocs),
    'question': RunnablePassthrough()
})

In [98]:
chain = parellel | promptTemplate | llm

In [99]:
chain.invoke("benifits of rag").content

'\n\nThe benefits of Retrieval Augmented Generation (RAG) include:  \n1. **High Accuracy and Reduced Hallucination**: By grounding responses in relevant source documents, RAG ensures answers are accurate and minimizes the risk of generating false information.  \n2. **Cost-Effectiveness**: RAG reduces token usage by passing only relevant context to the LLM, lowering API costs compared to sending entire documents.  \n\nThese benefits make RAG ideal for applications like medical knowledge assistants, legal compliance tools, and HR chatbots, where precision and efficiency are critical.'